In [ ]:
%%bash
nvidia-smi
lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core:" || true

In [ ]:
%%bash
# 1. Clean up old processes from prior runs
pkill -9 ollama || true
pkill -9 cloudflared || true

# 2. Install dependencies (zstd and PCI/GPU detection tools)
apt-get update -qq && apt-get install -y -qq zstd pciutils || sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils

# 3. Install Ollama
curl -fsSL https://ollama.com/install.sh | sh
ollama --version

In [ ]:
import os
import subprocess
import time
import requests

MODEL_NAME = "qwen3:8b"
CONTEXT_LENGTH = 32768
CPU_THREADS = 4

# Configure CPU (4-core) and Dual Tesla T4 GPU optimizations
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu:" + env.get("LD_LIBRARY_PATH", "")
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["OMP_NUM_THREADS"] = str(CPU_THREADS)
env["MKL_NUM_THREADS"] = str(CPU_THREADS)
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_FLASH_ATTENTION"] = "1"
env["OLLAMA_HOST"] = "0.0.0.0"
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_KEEP_ALIVE"] = "-1"  # Lock model in GPU VRAM permanently
env["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXT_LENGTH)

# Start Ollama daemon in background
ollama_log = open("/tmp/ollama.log", "w")
subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=ollama_log)

# Poll health check until port 11434 responds (up to 30s)
for attempt in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:11434/", timeout=2)
        if r.status_code == 200:
            print(f"Ollama server started with dual T4 GPU and {CPU_THREADS}-core CPU optimizations.")
            break
    except Exception:
        continue
else:
    with open("/tmp/ollama.log") as f:
        err = f.read()[-1000:]
    raise RuntimeError(f"Ollama failed to start within 30 seconds. Log tail:\n{err}")

In [ ]:
import requests

response = requests.get("http://127.0.0.1:11434/v1/models")
print("Ollama /v1/models status:", response.status_code)
print("Models payload:", response.json())

In [ ]:
%%bash
ollama pull qwen3:8b
ollama list

In [ ]:
import requests

# 1. Pre-warm model into GPU VRAM (pins model weights into dual T4 memory)
print("Pre-warming qwen3:8b into GPU VRAM (num_ctx: 32768, num_thread: 4)...")
prewarm_resp = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "qwen3:8b",
        "keep_alive": -1,
        "options": {
            "num_ctx": 32768,
            "num_thread": 4
        }
    },
    timeout=180
)
print("Pre-warm HTTP status:", prewarm_resp.status_code)

# 2. Local chat completion verification
test_resp = requests.post(
    "http://127.0.0.1:11434/v1/chat/completions",
    json={
        "model": "qwen3:8b",
        "messages": [{"role": "user", "content": "Say hello from Kaggle dual T4 GPU in one short sentence."}],
        "stream": False
    },
    timeout=60
)
print("Local test response:", test_resp.json()["choices"][0]["message"]["content"])

In [ ]:
%%bash
if ! command -v cloudflared &> /dev/null; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    sudo dpkg -i cloudflared-linux-amd64.deb || apt-get install -f -y
    rm -f cloudflared-linux-amd64.deb
fi
cloudflared --version

In [ ]:
import subprocess, time, re, requests, os, uuid

# Configure ntfy topic: check env, Kaggle secret, or generate a random session topic
NTFY_TOPIC = os.environ.get("NTFY_TOPIC")
if not NTFY_TOPIC:
    try:
        from kaggle_secrets import UserSecretsClient
        NTFY_TOPIC = UserSecretsClient().get_secret("NTFY_TOPIC")
    except Exception:
        pass
if not NTFY_TOPIC:
    NTFY_TOPIC = f"kaggle-ollama-{uuid.uuid4().hex[:12]}"

LOG_PATH = "/tmp/cloudflared.log"
URL_PATH = "/tmp/kaggle_tunnel_url.txt"
TOPIC_PATH = "/tmp/kaggle_ntfy_topic.txt"

# Terminate any prior tunnel process
subprocess.run("pkill -9 cloudflared || true", shell=True)

# Launch cloudflared tunnel in background
log_file = open(LOG_PATH, "w")
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print(f"cloudflared started (PID: {proc.pid})")

# Poll log file for trycloudflare URL
TUNNEL_URL = ""
pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
for _ in range(60):
    time.sleep(2)
    with open(LOG_PATH, errors="ignore") as f:
        m = pattern.search(f.read())
    if m:
        TUNNEL_URL = m.group(0)
        break
    if proc.poll() is not None:
        with open(LOG_PATH, errors="ignore") as f:
            err = f.read()[-1000:]
        raise RuntimeError(f"cloudflared exited early with code {proc.returncode}:\n{err}")

if not TUNNEL_URL:
    with open(LOG_PATH, errors="ignore") as f:
        err = f.read()[-1000:]
    raise RuntimeError(f"Timed out waiting for trycloudflare URL in /tmp/cloudflared.log:\n{err}")

print("TUNNEL_URL:", TUNNEL_URL)
with open(URL_PATH, "w") as f:
    f.write(TUNNEL_URL)
with open(TOPIC_PATH, "w") as f:
    f.write(NTFY_TOPIC)

# Post notification to ntfy.sh (retrievable in headless Kaggle API executions)
msg = f"Kaggle Ollama ready!\nEndpoint: {TUNNEL_URL}/v1\nModel: qwen3:8b\nKill switch: curl -d SHUTDOWN_GPU https://ntfy.sh/{NTFY_TOPIC}"
try:
    ntfy_resp = requests.post(f"https://ntfy.sh/{NTFY_TOPIC}", data=msg.encode(), timeout=30)
    print("ntfy post status:", ntfy_resp.status_code)
except Exception as e:
    print("ntfy post warning:", e)

print(f"ntfy topic url: https://ntfy.sh/{NTFY_TOPIC}")

In [ ]:
import requests, time, json

# Retrieve TUNNEL_URL with fallback
try:
    TUNNEL_URL
except NameError:
    with open("/tmp/kaggle_tunnel_url.txt") as f:
        TUNNEL_URL = f.read().strip()

print("Testing public endpoint:", TUNNEL_URL)

# 1. Verify models endpoint with DNS propagation tolerance
models_ok = False
for attempt in range(10):
    try:
        r = requests.get(f"{TUNNEL_URL}/v1/models", timeout=30)
        if r.status_code == 200:
            print(f"attempt {attempt+1} success: HTTP 200")
            models_ok = True
            break
    except Exception as e:
        print(f"attempt {attempt+1} waiting for DNS propagation (30s): {e}")
        time.sleep(30)

if not models_ok:
    print("WARNING: Public tunnel endpoint did not respond yet. Continuing to keep-alive...")
else:
    # 2. Verify Streaming (SSE) over Cloudflare tunnel
    print("Testing SSE streaming inference over public tunnel...")
    try:
        stream_resp = requests.post(
            f"{TUNNEL_URL}/v1/chat/completions",
            json={
                "model": "qwen3:8b",
                "messages": [{"role": "user", "content": "Hi"}],
                "stream": True
            },
            stream=True,
            timeout=60
        )
        first_chunk = next(stream_resp.iter_lines())
        print("SSE streaming verified! Sample chunk:", first_chunk.decode("utf-8")[:100])
    except Exception as e:
        print("WARNING: Streaming test encountered error:", e)

# 3. Auto-generate ready-to-copy Cline configuration
cline_config = {
    "apiKey": "none",
    "endpoint": f"{TUNNEL_URL}/v1",
    "model": {
        "contextWindow": 32768,
        "id": "qwen3:8b",
        "maxOutputTokens": 8192,
        "maxTokens": 32768,
        "name": "qwen3:8b"
    },
    "name": "ollama",
    "vendor": "ollama"
}

print("\n" + "="*60)
print("CLINE CONFIGURATION (Paste into VS Code Cline settings):")
print("="*60)
print(json.dumps(cline_config, indent=2))
print("="*60 + "\n")

In [ ]:
import time, datetime, requests, sys

try:
    TUNNEL_URL
except NameError:
    with open("/tmp/kaggle_tunnel_url.txt") as f:
        TUNNEL_URL = f.read().strip()

try:
    NTFY_TOPIC
except NameError:
    with open("/tmp/kaggle_ntfy_topic.txt") as f:
        NTFY_TOPIC = f.read().strip()

KEEP_ALIVE_HOURS = 4
CHECK_INTERVAL_SEC = 600  # 10 minutes
TOTAL_CHECKS = int(KEEP_ALIVE_HOURS * 3600 / CHECK_INTERVAL_SEC)

print(f"Keep-alive active ({KEEP_ALIVE_HOURS} hours, {TOTAL_CHECKS} heartbeats).")
print(f"Remote Kill Switch: send SHUTDOWN_GPU to https://ntfy.sh/{NTFY_TOPIC} to stop early and save quota.")

consecutive_failures = 0

for i in range(TOTAL_CHECKS):
    now = datetime.datetime.now().isoformat(timespec="seconds")
    
    # 1. Check for remote shutdown signal from ntfy
    try:
        poll_resp = requests.get(f"https://ntfy.sh/{NTFY_TOPIC}/json?poll=1&since=12m", timeout=10)
        if "SHUTDOWN_GPU" in poll_resp.text:
            print(f"[{now}] Remote kill switch received! Shutting down session safely...")
            sys.exit(0)
    except Exception:
        pass

    # 2. Local & Tunnel health check
    try:
        r = requests.get("http://127.0.0.1:11434/v1/models", timeout=20)
        if r.status_code == 200:
            consecutive_failures = 0
            print(f"[{now}] heartbeat {i+1}/{TOTAL_CHECKS} - healthy - endpoint: {TUNNEL_URL}")
        else:
            consecutive_failures += 1
            print(f"[{now}] heartbeat {i+1}/{TOTAL_CHECKS} - warning status {r.status_code}")
    except Exception as e:
        consecutive_failures += 1
        print(f"[{now}] heartbeat {i+1}/{TOTAL_CHECKS} - check failed ({consecutive_failures}/3): {e}")

    # 3. Crash protection: terminate kernel if service stays dead
    if consecutive_failures >= 3:
        print(f"[{now}] Ollama service failed 3 consecutive checks. Stopping session to conserve GPU quota.")
        sys.exit(1)

    if i < TOTAL_CHECKS - 1:
        time.sleep(CHECK_INTERVAL_SEC)

print("Keep-alive finished (4 hours). Shut the session down to save GPU quota.")

In [ ]:
%%bash
# Teardown and cleanup cell
pkill -9 ollama || true
pkill -9 cloudflared || true
rm -f /tmp/cloudflared.log /tmp/ollama.log /tmp/kaggle_tunnel_url.txt /tmp/kaggle_ntfy_topic.txt
echo "Cleanup completed. GPU processes stopped."